# Agent training

In [ ]:
from stable_baselines3 import PPO, A2C, DQN
from stable_baselines3.common.env_util import make_vec_env
from circle_environment import CircleEnv

# Instantiate the env
# vec_env = make_vec_env(CircleEnv, n_envs=1, env_kwargs=dict())

# env = CircleEnv(render_mode="human", log_level="info", vehicles_to_spawn=5)
env = CircleEnv(render_mode=None, vehicles_to_spawn=5)

In [ ]:
def configure_logging(self, log_file_path=None, console_log_level="info"):
    logger = logging.getLogger("application")
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    logger.setLevel(logging.DEBUG)
    if console_log_level is not None:
        # create console handler 
        ch = logging.StreamHandler()
        log_level = self.__convert_logging_level(console_log_level)
        ch.setLevel(log_level)
        ch.setFormatter(formatter)
        logger.addHandler(ch)
    if log_file_path is not None:
        # create file handler which logs even debug messages
        fh = logging.FileHandler(log_file_path, mode="a")
        fh.setLevel(logging.DEBUG)
        fh.setFormatter(formatter)
        logger.addHandler(fh)
    return logger

def __convert_logging_level(self, log_level_str):
    # Map string to logging level
    log_levels = {
        "debug": logging.DEBUG,
        "info": logging.INFO,
        "warning": logging.WARNING,
        "error": logging.ERROR,
        "critical": logging.CRITICAL
    }
    log_level = log_levels.get(log_level_str.lower(), logging.INFO)  # Default to INFO if not found
    return log_level

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback
from datetime import datetime
import os

# Create a unique identifier for this training run
current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_id = f"a2c_circle_v0.3_{current_time}"

# Set up directories
log_dir = f"./logs/{run_id}"
model_dir = f"./models"
os.makedirs(log_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

model_save_name = f"{model_dir}/{run_id}.zip"
tb_log_name = "tensorboard"
py_log_path = f"{log_dir}/{run_id}.log"
env.configure_logging(log_file_path=py_log_path, console_log_level=None)

class StepLoggerCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(StepLoggerCallback, self).__init__(verbose)
    
    def _on_step(self) -> bool:
        # TODO: num_timesteps an env weitergeben, damit der env-logger auch die sb3-timesteps loggen kann. Dann braucht der sb3-Logger das auch nicht mehr mitloggen.
        self.logger.record("current_step", self.num_timesteps)
        self.logger.dump(self.num_timesteps)
        return True


# Train the agent
model = A2C("MultiInputPolicy", env, verbose=1, tensorboard_log=log_dir)
learning_steps = 10000
model.learn(learning_steps, callback=StepLoggerCallback(), tb_log_name=tb_log_name)
model.save(model_save_name)

In [ ]:
# Quick evaluation
from stable_baselines3.common.evaluation import evaluate_policy
print("Training finished. Starting evaluation")
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=1)
print(mean_reward)
print(std_reward)

In [ ]:
# Cleanup
env.close()

In [ ]:
from stable_baselines3 import PPO, A2C, DQN
from gymtest import CircleEnv

# Observe execution of trained agent in GUI
env = CircleEnv(render_mode="human", log_level="info", vehicles_to_spawn=5)
model = A2C.load(model_save_name, env)

num_steps = 5
observation, info = env.reset()
for t in range(num_steps):
        actions, _ = model.predict(observation, state=None, deterministic=False)
        observation, reward, terminated, truncated, info = env.step(actions)

env.close()

In [ ]:
# Random actions to compare with the agent
env = CircleEnv(render_mode="human", log_level="info", vehicles_to_spawn=5)
observation, info = env.reset()
for _ in range(5):
    action = env.action_space.sample() # select a random action
    observation, reward, terminated, truncated, info = env.step(action)
    # if terminated or truncated:
        # observation, info = env.reset()
        
env.close()

---
# Test charging stop removal

In [ ]:
from circletest import Simulation

cs_id = "cs_0"
vehicle_id = "myVehicle0"
simulation = Simulation(gui=False)
simulation.add_vehicles()

def print_stops():
    stops = simulation.get_stops(vehicle_id)
    print(f"Stops: {stops}")

# starten
simulation.step()
print_stops()
# rerouten
print("REROUTE")
simulation.reroute_for_charging(vehicle_id, cs_id)
print_stops()
# stop removen
print("REMOVE STOP")
simulation.remove_charging_stop(vehicle_id)
print_stops()

simulation.close()